# Lab 04.1 — Apache Ozone come storage per Spark

## Introduzione
**Ozone** è un object store distribuito del mondo Hadoop, pensato per miliardi di oggetti (dove HDFS soffre per i metadati in memoria sul NameNode).
In questo notebook usiamo Ozone come storage per Spark, attraverso il filesystem Hadoop `ofs://`.

| Componente | Ruolo |
|---|---|
| **Ozone Manager (OM)** | namespace: volumi, bucket, chiavi |
| **Storage Container Manager (SCM)** | blocchi, *container* e repliche sui DataNode |
| **DataNode** | memorizzano i dati |
| **S3 Gateway** | espone i bucket con l'API Amazon S3 |

Gerarchia dei dati: **volume → bucket → chiave** (es. `/lab/vendite/2024/file.parquet`).
Spark accede a Ozone con **`ofs://`** (*Rooted Ozone FileSystem*): `ofs://<om>/<volume>/<bucket>/<chiave>`.
Il client (`ozone-filesystem-hadoop3.jar`) è già incluso nell'immagine Spark.

UI: Ozone Manager http://localhost:9874 · SCM http://localhost:9876

### Obiettivi
1. Configurare una `SparkSession` che accede a Ozone tramite `ofs://`.
2. Creare volumi e bucket con l'API FileSystem di Hadoop.
3. Caricare un file e fare un Word Count direttamente da Ozone.
4. Scrivere e interrogare un DataFrame Parquet partizionato.
5. Scrivere altri formati (CSV, JSON) sullo stesso bucket.
6. Verificare la rinomina atomica, resa possibile dal layout `FILE_SYSTEM_OPTIMIZED`.

## Step 1: Inizializzare la SparkSession con `ofs://`

Spark gira in modalità `local[*]` dentro Jupyter: il laboratorio riguarda lo storage, non il calcolo distribuito.
Servono tre configurazioni (il prefisso `spark.hadoop.` le passa alla configurazione Hadoop):

| Opzione | Significato |
|---|---|
| `fs.ofs.impl` | classe che implementa lo schema `ofs://` |
| `ozone.om.address` | indirizzo RPC dell'Ozone Manager (`om`, porta 9862) |
| `ozone.replication` | numero di repliche: 1, perché c'è un solo DataNode |

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("Lab04-Ozone")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         # Implementazione del filesystem ofs:// e indirizzo dell'Ozone Manager
         .config("spark.hadoop.fs.ofs.impl", "org.apache.hadoop.fs.ozone.RootedOzoneFileSystem")
         .config("spark.hadoop.ozone.om.address", "om:9862")
         # Un solo DataNode nel laboratorio → una sola replica
         .config("spark.hadoop.ozone.replication", "1")
         .getOrCreate())
sc = spark.sparkContext
# Radice del filesystem Ozone: ofs://<om>/<volume>/<bucket>/<chiave>
OZ = "ofs://om"
print("Master:", sc.master)

## Step 2: Volumi e bucket con l'API FileSystem

Con `ofs://` il primo livello del percorso è il **volume**, il secondo il **bucket**: si creano semplicemente come directory con `mkdirs`.
Accediamo all'API Java di Hadoop attraverso la JVM di Spark (`sc._jvm`) e definiamo la funzione `ls` per elencare il contenuto di un percorso.
Il volume `lab` conterrà i bucket `vendite` e `testi`.

In [ ]:
# Accesso all'API Java di Hadoop attraverso la JVM di Spark
jvm = sc._jvm
Path = jvm.org.apache.hadoop.fs.Path
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI(f"{OZ}/"), sc._jsc.hadoopConfiguration())

fs.mkdirs(Path(f"{OZ}/lab/vendite"))      # volume "lab", bucket "vendite"
fs.mkdirs(Path(f"{OZ}/lab/testi"))        # bucket "testi"

# Elenca il contenuto di un percorso: tipo (d = directory), dimensione, percorso
def ls(percorso):
    for st in fs.listStatus(Path(percorso)):
        tipo = "d" if st.isDirectory() else "-"
        print(f"{tipo} {st.getLen():>12,d}  {st.getPath()}")

print("Volumi:");             ls(f"{OZ}/")
print("\nBucket del volume lab:"); ls(f"{OZ}/lab")

### Punti chiave
- Ozone organizza i dati in **volume → bucket → chiave**; con `ofs://` volumi e bucket si creano come directory.
- Spark usa Ozone come un normale filesystem Hadoop: basta configurare `fs.ofs.impl` e l'indirizzo dell'OM.

## Step 3: Caricare un file e fare Word Count

`copyFromLocalFile` scrive una chiave nel bucket `testi` copiando un file dal dataset condiviso (montato in sola lettura in `/home/jovyan/datasets`).
Spark poi legge la chiave in parallelo come un normale file Hadoop: tokenizza il testo, scarta le parole corte (3 lettere o meno), conta le occorrenze con `reduceByKey` e restituisce le 8 più frequenti.

In [ ]:
# Copia il file locale in una chiave del bucket "testi" e ne mostra il contenuto
fs.copyFromLocalFile(Path("file:///home/jovyan/datasets/alice_in_wonderland.txt"), Path(f"{OZ}/lab/testi/alice.txt"))
ls(f"{OZ}/lab/testi")

import re
# Word Count classico (RDD) leggendo direttamente da Ozone
top = (sc.textFile(f"{OZ}/lab/testi/alice.txt")
         .flatMap(lambda l: re.findall(r"[a-z]+", l.lower()))   # parole in minuscolo
         .filter(lambda w: len(w) > 3)                           # scarta le parole corte
         .map(lambda w: (w, 1))
         .reduceByKey(lambda a, b: a + b)                        # somma per parola
         .takeOrdered(8, key=lambda kv: -kv[1]))                 # le 8 più frequenti
print(top)

### Punti chiave
- Un file caricato in Ozone diventa una **chiave** del bucket, leggibile da Spark con `textFile` o `spark.read`.
- Il codice Spark non cambia rispetto a HDFS o al disco locale: cambia solo lo schema del percorso (`ofs://`).

## Step 4: DataFrame e Parquet partizionato su Ozone

Generiamo 500.000 vendite sintetiche con `anno` (2023-2025), `prodotto` e `importo`, e le scriviamo in **Parquet** con `partitionBy("anno")`: ogni anno diventa una sottodirectory `anno=...` del bucket.
Il seed fisso in `rand(11)` rende i dati riproducibili.

In [ ]:
# Dati sintetici: 500.000 vendite su 3 anni e 50 prodotti
vendite = (spark.range(0, 500_000)
           .withColumn("anno", (2023 + F.col("id") % 3).cast("int"))
           .withColumn("prodotto", F.concat(F.lit("P"), (F.col("id") % 50).cast("string")))
           .withColumn("importo", F.round(F.rand(11) * 200, 2)))

# Scrittura in Parquet partizionato per anno (una sottodirectory per anno)
vendite.write.partitionBy("anno").mode("overwrite").parquet(f"{OZ}/lab/vendite/parquet")
ls(f"{OZ}/lab/vendite/parquet")

### Step 4b: Interrogare il Parquet con Spark SQL

Rileggiamo il dataset da Ozone e lo registriamo come vista temporanea `vendite`.
Il filtro `anno >= 2024` agisce sulla colonna di partizione: Spark legge solo le directory `anno=2024` e `anno=2025` (*partition pruning*).

In [ ]:
# Vista temporanea sul Parquet letto da Ozone
spark.read.parquet(f"{OZ}/lab/vendite/parquet").createOrReplaceTempView("vendite")
# Il filtro sulla colonna di partizione evita di leggere la partizione 2023
spark.sql("""
    SELECT anno, COUNT(*) AS n, CAST(SUM(importo) AS DECIMAL(14, 2)) AS totale
    FROM vendite
    WHERE anno >= 2024
    GROUP BY anno ORDER BY anno
""").show()

### Punti chiave
- Il Parquet partizionato si scrive e si legge su Ozone esattamente come su HDFS.
- Il *partition pruning* riduce i dati letti dallo storage: conviene partizionare sulle colonne usate nei filtri.

## Step 5: Altri formati sullo stesso bucket

Calcoliamo i 10 prodotti con il maggior incasso e li salviamo in **CSV** e **JSON** nello stesso bucket.
`coalesce(1)` produce un solo file per formato (il risultato è piccolo); poi rileggiamo il JSON per verifica.

In [ ]:
# I 10 prodotti con il maggior incasso totale
top_prodotti = spark.sql("""
    SELECT prodotto, ROUND(SUM(importo), 2) AS totale
    FROM vendite GROUP BY prodotto ORDER BY totale DESC LIMIT 10
""")
# coalesce(1): un solo file di output per formato
top_prodotti.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OZ}/lab/vendite/report_csv")
top_prodotti.coalesce(1).write.mode("overwrite").json(f"{OZ}/lab/vendite/report_json")
ls(f"{OZ}/lab/vendite")
spark.read.json(f"{OZ}/lab/vendite/report_json").show(3)

## Step 6: Rinomina atomica

I bucket creati tramite `ofs://` hanno layout **FILE_SYSTEM_OPTIMIZED**: le directory esistono davvero e `rename` è un'operazione sui metadati
(non una copia, come accade negli object store puri). È ciò che rende sicuri i *commit* dei job Spark.
Prima eliminiamo l'eventuale destinazione, così la cella si può rieseguire; poi rinominiamo `report_csv` in `report_finale_csv`.

In [ ]:
# Elimina la destinazione se esiste già (il notebook può essere rieseguito)
fs.delete(Path(f"{OZ}/lab/vendite/report_finale_csv"), True)
# Rinomina atomica: operazione sui soli metadati, nessuna copia dei dati
print("rename:", fs.rename(Path(f"{OZ}/lab/vendite/report_csv"), Path(f"{OZ}/lab/vendite/report_finale_csv")))
ls(f"{OZ}/lab/vendite")

### Punti chiave
- Su Ozone si possono scrivere CSV, JSON, Parquet nello stesso bucket, con le stesse API di Spark.
- Con layout `FILE_SYSTEM_OPTIMIZED` la rinomina è atomica e veloce: i commit dei job non richiedono copie.

## Step 7: Chiudere la SparkSession

Liberiamo le risorse del driver. I dati restano su Ozone e sono usati anche dal notebook successivo.

In [ ]:
spark.stop()

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Accesso a Ozone | `fs.ofs.impl`, `ozone.om.address`, `ofs://om/...` | leggere e scrivere su Ozone da Spark |
| Volumi e bucket | `fs.mkdirs(Path("ofs://om/vol/bucket"))` | creare il namespace |
| Caricare file | `fs.copyFromLocalFile` | scrivere una chiave dal disco locale |
| Word Count | `sc.textFile` + `reduceByKey` | elaborare in parallelo un file su Ozone |
| Parquet partizionato | `write.partitionBy("anno").parquet(...)` | layout colonnare con partition pruning |
| Altri formati | `write.csv(...)`, `write.json(...)` | esportare risultati sullo stesso bucket |
| Rinomina atomica | `fs.rename` (layout `FILE_SYSTEM_OPTIMIZED`) | commit sicuri dei job senza copie |